# Milestone reconciliation

**Purpose (PRD §13):** the pipeline's `get_success_milestone` computes the
brief's success-milestone definition straight from `research.db`. Before
trusting that output, it needs to agree with the milestone table that was
originally built by hand. This notebook loads both, compares them side by
side, and flags every title where they disagree — per the PRD, a
disagreement here has to end in either an explanation or a correction, not
a shrug.

**Scope:** every *actively tracked* title whose Liquipedia crawl is
complete — see "Confirm crawl completeness" below. That's a check, not a
filter: as of this notebook's first version, every active title's crawl is
complete, so nothing is excluded today. If a future title is added to
`config/titles.yaml` and its crawl hasn't caught up yet (or hits the kind
of known gap `collectors/liquipedia.py`'s docstring describes), that check
will fail loudly instead of silently comparing against incomplete data.

In [1]:
# Imports and repo path setup.
#
# Jupyter runs notebooks with the notebook's own directory as the working
# directory, but this project's modules (etl.db, analysis.metrics) are
# imported as `repo_root.module`, the same way collectors/liquipedia.py
# does it. So: find the repo root (one level up from notebooks/) and put it
# on sys.path before importing anything project-local.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml

from analysis.metrics import get_success_milestone
from etl.db import get_connection

In [2]:
# Load the tracked-title list. config/titles.yaml is this project's single
# source of truth for which titles exist (CLAUDE.md) — we need it here for
# the id -> display-name mapping, and to know which titles to check below.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
len(active_titles), list(display_name_by_id)[:5]

(23,
 ['league_of_legends', 'dota2', 'counter_strike', 'starcraft2', 'hearthstone'])

## Confirm crawl completeness

"Complete" means every active title has at least one row in `tournaments`
— i.e. `collectors/liquipedia.py` found a recognized tier convention for
its wiki (possibly narrowed by `liquipedia_category`, for the shared
fighting-game wiki) and wrote real data, rather than logging it as skipped
(see that module's docstring). A title with zero rows has nothing for
`get_success_milestone` to work with, so comparing it against the
hand-built table would produce a false "pipeline says never reached"
rather than a real disagreement — this check exists to catch that before
it happens silently.

In [3]:
# Count tournament rows per active title and fail loudly if any are at
# zero, instead of letting them flow into the comparison as a false
# "pipeline found nothing" result.
conn = get_connection()
row_counts = dict(conn.execute("SELECT title_id, COUNT(*) FROM tournaments GROUP BY title_id").fetchall())
conn.close()

incomplete = [t["id"] for t in active_titles if row_counts.get(t["id"], 0) == 0]
if incomplete:
    raise RuntimeError(
        f"{len(incomplete)} active title(s) have no tournament data yet, so "
        f"this notebook can't reconcile them: {incomplete}. Run "
        "`python collectors/liquipedia.py --titles " + ",".join(incomplete) + "` "
        "(or address whatever collectors/liquipedia.py logged as the reason "
        "it skipped them) before re-running this notebook."
    )

print(f"all {len(active_titles)} active titles have tournament data — crawl is complete.")

all 23 active titles have tournament data — crawl is complete.


In [4]:
# Compute the pipeline's milestone for every active title.
#
# get_success_milestone reads straight from the `tournaments` table (i.e.
# whatever collectors/liquipedia.py has crawled so far) and returns a dict
# per title; get_connection() opens research.db read/write (WAL mode, so
# this doesn't block a collector run happening at the same time) but we
# only read here.
conn = get_connection()
pipeline_rows = [get_success_milestone(conn, t["id"]) for t in active_titles]
conn.close()

pipeline_df = pd.DataFrame(pipeline_rows)
pipeline_df["display_name"] = pipeline_df["title_id"].map(display_name_by_id)
pipeline_df = pipeline_df[["title_id", "display_name", "milestone_year"]]
pipeline_df = pipeline_df.rename(columns={"milestone_year": "milestone_year_pipeline"})
pipeline_df

,title_id,display_name,milestone_year_pipeline
0,league_of_legends,League of Legends,2011.0
1,dota2,Dota 2,2006.0
2,counter_strike,Counter-Strike 2,NaN
3,starcraft2,StarCraft II,2011.0
4,hearthstone,Hearthstone,2014.0
5,rocket_league,Rocket League,2016.0
6,rainbow_six_siege,Rainbow Six Siege,2017.0
7,overwatch,Overwatch,2017.0
8,tekken,Tekken 8,2026.0
9,street_fighter,Street Fighter 6,2024.0


## Load the hand-built milestone table

This is the independent test fixture PRD §13 calls for: numbers produced
*before* this pipeline existed, so they don't just reflect the same bugs
back. It isn't in the repo yet — see `data/manual/README.md` for the
expected format. Short version: a CSV at `data/manual/milestone_table.csv`
with one row per title (`title_id`, `milestone_year`, optional `notes`),
`title_id` matching `config/titles.yaml`'s ids exactly, one row for every
title — including ones your original table didn't consider to have
reached the milestone (blank `milestone_year`), since this notebook needs
to tell "hand-built says not yet" apart from "hand-built has no opinion at
all."

If you don't already have `title_id` slugs attached to your original
table, match them by `display_name` against the table two cells up and
copy the ids across — that's the only manual mapping step.

In [5]:
# Load the hand-built table, with a guard that explains exactly what's
# missing rather than a bare FileNotFoundError, since this is the one
# input this notebook can't derive on its own.
MANUAL_PATH = REPO_ROOT / "data" / "manual" / "milestone_table.csv"

if not MANUAL_PATH.exists():
    raise FileNotFoundError(
        f"Expected your hand-built milestone table at {MANUAL_PATH}, "
        "but it isn't there yet.\n\n"
        "Create it as a CSV with one row per title and these columns:\n"
        "  title_id        - must match an id in config/titles.yaml\n"
        "  milestone_year  - year your original table says the milestone "
        "was reached; blank if it wasn't\n"
        "  notes           - optional, free text\n\n"
        "See data/manual/README.md for the full spec, and the "
        "'display_name_by_id' mapping above if your table used display "
        "names instead of ids."
    )

manual_df = pd.read_csv(MANUAL_PATH, dtype={"title_id": str})
manual_df = manual_df.rename(columns={"milestone_year": "milestone_year_manual"})
manual_df.head()

,title_id,milestone_year_manual,notes
0,age_of_empires_ii,2023,In our window of analysis (up to about Dec '23...
1,apex_legends,2023,https://escharts.com/games/apex
2,brawl_stars,2022,Flip to the Viewership Over Time tab to see th...
3,counter_strike,2015,https://esl.com/article/esl-one-cologne-2015-t...
4,dota2,2015,https://liquipedia.net/dota2/Tier_1_Tournament...


## Side-by-side comparison

Left join from `pipeline_df` (every active title) onto the hand-built
table, so every tracked title appears exactly once, with whatever the
hand-built table says for it (or nothing, if that title isn't in the
hand-built table at all — itself worth flagging, not just skipping).

In [6]:
# Join pipeline and hand-built figures for the same set of titles. Track
# which titles have a row in the hand-built table at all, separately from
# what that row's milestone_year says — a blank year ("not reached yet")
# and a missing row ("no opinion") mean different things below.
titles_in_manual = set(manual_df["title_id"])

comparison = pipeline_df.merge(
    manual_df[["title_id", "milestone_year_manual"]], on="title_id", how="left"
)
comparison["in_manual_table"] = comparison["title_id"].isin(titles_in_manual)
comparison

,title_id,display_name,milestone_year_pipeline,milestone_year_manual,in_manual_table
0,league_of_legends,League of Legends,2011.0,2014.0,True
1,dota2,Dota 2,2006.0,2015.0,True
2,counter_strike,Counter-Strike 2,NaN,2015.0,True
3,starcraft2,StarCraft II,2011.0,2013.0,True
4,hearthstone,Hearthstone,2014.0,2015.0,True
5,rocket_league,Rocket League,2016.0,2018.0,True
6,rainbow_six_siege,Rainbow Six Siege,2017.0,2020.0,True
7,overwatch,Overwatch,2017.0,2019.0,True
8,tekken,Tekken 8,2026.0,2018.0,True
9,street_fighter,Street Fighter 6,2024.0,NaN,False


In [7]:
# Flag every disagreement.
#   - no row at all in the hand-built table -> MISSING_IN_MANUAL
#   - both sides agree the title hasn't reached the milestone (both years
#     blank) -> MATCH
#   - both sides have a year and it's the same -> MATCH
#   - anything else (one side has a year and the other doesn't, or they
#     have different years) -> MISMATCH
def classify(row):
    if not row["in_manual_table"]:
        return "MISSING_IN_MANUAL"
    pipeline_year = row["milestone_year_pipeline"]
    manual_year = row["milestone_year_manual"]
    if pd.isna(pipeline_year) and pd.isna(manual_year):
        return "MATCH"
    if pd.isna(pipeline_year) or pd.isna(manual_year):
        return "MISMATCH"
    return "MATCH" if int(pipeline_year) == int(manual_year) else "MISMATCH"

comparison["status"] = comparison.apply(classify, axis=1)
comparison

,title_id,display_name,milestone_year_pipeline,milestone_year_manual,in_manual_table,status
0,league_of_legends,League of Legends,2011.0,2014.0,True,MISMATCH
1,dota2,Dota 2,2006.0,2015.0,True,MISMATCH
2,counter_strike,Counter-Strike 2,NaN,2015.0,True,MISMATCH
3,starcraft2,StarCraft II,2011.0,2013.0,True,MISMATCH
4,hearthstone,Hearthstone,2014.0,2015.0,True,MISMATCH
5,rocket_league,Rocket League,2016.0,2018.0,True,MISMATCH
6,rainbow_six_siege,Rainbow Six Siege,2017.0,2020.0,True,MISMATCH
7,overwatch,Overwatch,2017.0,2019.0,True,MISMATCH
8,tekken,Tekken 8,2026.0,2018.0,True,MISMATCH
9,street_fighter,Street Fighter 6,2024.0,NaN,False,MISSING_IN_MANUAL


In [8]:
# The actual output this notebook exists to produce: anything that isn't a
# clean MATCH, per PRD §13 ("every discrepancy either explained or
# corrected") — each of these needs a decision recorded somewhere (a fix to
# the pipeline, a correction to config/titles.yaml, or a note added to
# data/manual/milestone_table.csv's `notes` column), not just an
# acknowledgment here.
disagreements = comparison[comparison["status"] != "MATCH"]

print(f"{len(disagreements)} of {len(comparison)} titles disagree.")
disagreements

23 of 23 titles disagree.


,title_id,display_name,milestone_year_pipeline,milestone_year_manual,in_manual_table,status
0,league_of_legends,League of Legends,2011.0,2014.0,True,MISMATCH
1,dota2,Dota 2,2006.0,2015.0,True,MISMATCH
2,counter_strike,Counter-Strike 2,NaN,2015.0,True,MISMATCH
3,starcraft2,StarCraft II,2011.0,2013.0,True,MISMATCH
4,hearthstone,Hearthstone,2014.0,2015.0,True,MISMATCH
5,rocket_league,Rocket League,2016.0,2018.0,True,MISMATCH
6,rainbow_six_siege,Rainbow Six Siege,2017.0,2020.0,True,MISMATCH
7,overwatch,Overwatch,2017.0,2019.0,True,MISMATCH
8,tekken,Tekken 8,2026.0,2018.0,True,MISMATCH
9,street_fighter,Street Fighter 6,2024.0,NaN,False,MISSING_IN_MANUAL


## Context for each disagreement

`get_success_milestone` (2026-09-05) now also returns `viewership_check`
(the brief's "flat or growing" clause, evaluated against whatever data
source has coverage — see that function's docstring) and
`qualifying_window_scale` (absolute prize-pool/team-scale figures for the
qualifying window, a permanent companion to the milestone flag rather than
a gating threshold — PRD §6). Neither changes `milestone_year` itself, but
both are exactly the kind of context PRD §9's "explained or corrected" bar
calls for, so they're surfaced here per disagreement rather than left
sitting unused in the pipeline's return value.

In [9]:
conn = get_connection()
pipeline_by_id = {t["id"]: get_success_milestone(conn, t["id"]) for t in active_titles}
conn.close()


def summarize_viewership(title_id):
    check = pipeline_by_id[title_id]["viewership_check"]
    if check is None:
        return "not evaluable (no source has coverage for this window)"
    trend = "flat/growing" if check["flat_or_growing"] else "declining"
    return f"{trend} ({check['source']}, {check['confidence']}, esports_specific={check['esports_specific']})"


def summarize_scale(title_id):
    scale = pipeline_by_id[title_id]["qualifying_window_scale"]
    if scale is None:
        return "n/a (no milestone found)"
    prize = scale["prize_pool_by_currency"]
    prize_str = ", ".join(f"{cur} {v['total']:,.0f}" for cur, v in prize.items()) or "no prize data"
    team = f"{scale['avg_team_number']:.1f} avg teams" if scale["avg_team_number"] is not None else "no team data"
    return f"{scale['tournament_count']} tournaments, {team}, {prize_str}"


context_df = disagreements.copy()
context_df["window_start_year"] = context_df["title_id"].map(lambda t: pipeline_by_id[t]["window_start_year"])
context_df["viewership_check"] = context_df["title_id"].map(summarize_viewership)
context_df["qualifying_window_scale"] = context_df["title_id"].map(summarize_scale)
context_df[
    [
        "title_id",
        "display_name",
        "window_start_year",
        "milestone_year_pipeline",
        "milestone_year_manual",
        "status",
        "viewership_check",
        "qualifying_window_scale",
    ]
]

,title_id,display_name,window_start_year,milestone_year_pipeline,milestone_year_manual,status,viewership_check,qualifying_window_scale
0,league_of_legends,League of Legends,2010.0,2011.0,2014.0,MISMATCH,not evaluable (no source has coverage for this...,"12 tournaments, 12.0 avg teams, USD 238,500, c..."
1,dota2,Dota 2,2005.0,2006.0,2015.0,MISMATCH,not evaluable (no source has coverage for this...,"8 tournaments, 30.6 avg teams, rub 85,000, SGD..."
2,counter_strike,Counter-Strike 2,NaN,NaN,2015.0,MISMATCH,not evaluable (no source has coverage for this...,n/a (no milestone found)
3,starcraft2,StarCraft II,2010.0,2011.0,2013.0,MISMATCH,not evaluable (no source has coverage for this...,"130 tournaments, 8.7 avg teams, unknown 1,602,..."
4,hearthstone,Hearthstone,2013.0,2014.0,2015.0,MISMATCH,not evaluable (no source has coverage for this...,"57 tournaments, 33.5 avg teams, USD 731,175, C..."
5,rocket_league,Rocket League,2015.0,2016.0,2018.0,MISMATCH,"flat/growing (monthly_category_history, proxy_...","15 tournaments, 503.0 avg teams, USD 306,000, ..."
6,rainbow_six_siege,Rainbow Six Siege,2016.0,2017.0,2020.0,MISMATCH,"flat/growing (monthly_category_history, proxy_...","45 tournaments, 12.1 avg teams, USD 1,097,000"
7,overwatch,Overwatch,2016.0,2017.0,2019.0,MISMATCH,"declining (monthly_category_history, proxy_est...","105 tournaments, 29.8 avg teams, USD 2,108,140..."
8,tekken,Tekken 8,2025.0,2026.0,2018.0,MISMATCH,not evaluable (no source has coverage for this...,"6 tournaments, no team data, unknown 400,000, ..."
9,street_fighter,Street Fighter 6,2023.0,2024.0,NaN,MISSING_IN_MANUAL,not evaluable (no source has coverage for this...,"25 tournaments, 7.5 avg teams, unknown 1,244,1..."
